# 01 - Thẩm Định Dữ Liệu (EDA) & Thiết Lập Data Pipeline
**Đồ án:** Chẩn đoán bệnh trên lá cây nông nghiệp (PlantVillage Dataset)
**Tác giả:** Senior ML Engineer & Mentor

### Nguyên lý thiết kế hệ thống:
- **Designing Machine Learning Systems (Chip Huyen)**: Chương 3, 4, 5 (Data Engineering, Metrics Selection, Data Leakage Prevention).
- **Hands-On Machine Learning (Aurélien Géron)**: Chương 2, 13, 14 (End-to-End Project, Data Pipelines, Deep Computer Vision).

---
### 1. Mục tiêu
1. Kiểm tra tính toàn vẹn (Data Integrity) trên 38 lớp cây trồng & bệnh lý.
2. Phân tích phân phối nhãn (Class Distribution) & tính Imbalance Ratio.
3. Xây dựng Data Augmentation Pipeline phòng chống Data Leakage.
4. Kiểm thử DataLoader sẵn sàng cho Baseline 1 (Vanilla CNN), Baseline 2 (MobileNetV2), Baseline 3 (ResNet50).

In [ ]:
import os
import json
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import datasets, transforms
from data_pipeline import build_dataloaders, get_train_transforms, get_eval_transforms

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

### 2. Đọc Báo Cáo Phân Phối Nhãn Đã Trích Xuất

In [ ]:
df_dist = pd.read_csv("eda_reports/class_distribution.csv")
print(f"Tổng số lớp bệnh/cây: {len(df_dist)}")
print(f"Tổng số ảnh tập train: {df_dist['train_count'].sum():,}")
print(f"Tổng số ảnh tập valid: {df_dist['valid_count'].sum():,}")
df_dist.head(10)

### 3. Trực Quan Hóa Mẫu Ảnh Gốc và Ảnh sau Augmentation
*Minh họa tư duy Hands-On ML (Chương 14): Data Augmentation mở rộng biên quyết định (decision boundary) và ngăn mô hình học vẹt phông nền nông trường.*

In [ ]:
# Lấy 1 ảnh mẫu từ tập train để quan sát hiệu ứng Augmentation
sample_class = df_dist.iloc[0]["class_name"]
sample_img_path = list((Path("../data/train") / sample_class).glob("*.JPG"))[0]
orig_img = Image.open(sample_img_path)

train_tf = get_train_transforms(image_size=224)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(orig_img)
axes[0].set_title("Ảnh gốc (256x256)")
axes[0].axis("off")

for i in range(1, 5):
    augmented = train_tf(orig_img)
    # Denormalize để hiển thị chuẩn RGB
    img_disp = augmented.clone()
    for c, mean, std in zip(img_disp, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]):
        c.mul_(std).add_(mean)
    img_disp = img_disp.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[i].imshow(img_disp)
    axes[i].set_title(f"Augmented #{i}")
    axes[i].axis("off")
plt.suptitle(f"Demo Data Augmentation: {sample_class}", fontsize=14, fontweight="bold")
plt.show()

### 4. Kiểm Thử DataLoader & Tải Dữ Liệu Sẵn Sàng Cho Huấn Luyện
*Thiết lập shuffle=True cho train, shuffle=False cho valid; áp dụng class weights cho CrossEntropyLoss.*

In [ ]:
train_loader, valid_loader, idx_to_class, class_weights = build_dataloaders(
    data_dir="../data",
    batch_size=64,
    image_size=224,
    num_workers=4,
    pin_memory=False
)

print(f"Batches Train: {len(train_loader)} | Batches Valid: {len(valid_loader)}")
images, labels = next(iter(train_loader))
print(f"Shape Batch Ảnh: {images.shape}")
print(f"Shape Batch Nhãn: {labels.shape}")
print(f"Min / Max Pixel Values: {images.min():.2f} / {images.max():.2f}")
print(f"Class Weights: Shape {class_weights.shape}, Min={class_weights.min():.3f}, Max={class_weights.max():.3f}")